# 02 — Cleaning & Feature Engineering

Input: `philly311_raw.csv` from notebook 01.

**Before running:** open `philly311_raw.csv` and Step 2 of notebook 01's output, and fix
the column names in the CONFIG cell below so they match reality. Everything downstream
references these variables, so getting this right once saves you from hunting through
every cell later.

In [ ]:
import pandas as pd
import numpy as np

# ---- CONFIG: confirm/edit these against your actual columns ----
CREATED_COL = "requested_datetime"   # candidates: requested_datetime, created_date
CLOSED_COL  = "closed_datetime"      # candidates: closed_datetime, closed_date
TYPE_COL    = "service_name"         # the request-type column you used in notebook 01
STATUS_COL  = "status"
ZIP_COL     = "zipcode"
AGENCY_COL  = "agency_responsible"
LAT_COL     = "lat"
LON_COL     = "lon"
ID_COL      = "objectid"             # or service_request_id, whichever is the unique id

df = pd.read_csv("philly311_raw.csv")
print(df.shape)
df.head()

## Step A — Parse dates and compute resolution time

In [ ]:
df[CREATED_COL] = pd.to_datetime(df[CREATED_COL], errors="coerce")
df[CLOSED_COL] = pd.to_datetime(df[CLOSED_COL], errors="coerce")

df["resolution_hours"] = (df[CLOSED_COL] - df[CREATED_COL]).dt.total_seconds() / 3600

print("Rows with no created date:", df[CREATED_COL].isna().sum())
print("Rows with no closed date (i.e. still open, or never closed in this export):", df[CLOSED_COL].isna().sum())
print("Rows with negative resolution time (data errors):", (df["resolution_hours"] < 0).sum())

## Step B — Decide how to handle open/invalid requests

Default policy below (edit and justify in your report if you change it):
- Drop rows with no created date (can't use them for anything)
- Drop rows with negative resolution time (data entry errors)
- Keep still-open rows in a *separate* dataframe rather than silently dropping them —
  they matter for context (e.g. "12% of requests are still open after 2 years") even
  though they can't be used as labeled examples for regression/classification.

In [ ]:
df = df[df[CREATED_COL].notna()].copy()

still_open = df[df[CLOSED_COL].isna()].copy()
print(f"Still-open requests set aside: {len(still_open)} ({len(still_open)/len(df):.1%} of data)")

df = df[df[CLOSED_COL].notna()].copy()
n_before = len(df)
df = df[df["resolution_hours"] >= 0].copy()
print(f"Dropped {n_before - len(df)} rows with negative resolution time")
print(f"Remaining rows for modeling: {len(df)}")

## Step C — Duplicates and missing key fields

In [ ]:
dupes = df.duplicated(subset=[ID_COL]).sum()
print(f"Duplicate IDs: {dupes}")
df = df.drop_duplicates(subset=[ID_COL])

for col in [ZIP_COL, AGENCY_COL, LAT_COL, LON_COL]:
    if col in df.columns:
        pct_missing = df[col].isna().mean()
        print(f"{col}: {pct_missing:.1%} missing")

## Step D — Feature engineering

In [ ]:
df["day_of_week"] = df[CREATED_COL].dt.day_name()
df["is_weekend"] = df[CREATED_COL].dt.dayofweek >= 5
df["hour_of_day"] = df[CREATED_COL].dt.hour
df["month"] = df[CREATED_COL].dt.month
df["year"] = df[CREATED_COL].dt.year

# log-transform target given the expected right skew (confirm this is warranted in notebook 03)
df["log_resolution_hours"] = np.log1p(df["resolution_hours"])

df.head()

## Step E — Save the clean dataset

In [ ]:
df.to_csv("philly311_clean.csv", index=False)
still_open.to_csv("philly311_still_open.csv", index=False)
print("Saved philly311_clean.csv and philly311_still_open.csv")

## Write these down for your report
- How many rows you started with vs. ended with, and why each drop happened
- % still-open, and your reasoning for setting them aside
- Any missingness you noticed and what you did about it
- Anything surprising you saw while doing this (this is exactly the kind of thing your
  professor is asking about in "what problems did you encounter and how did you solve them")